# Entraînement du modèle YOLOv11

## Environnement et dépendances

In [ ]:
import os
from pathlib import Path
import mlflow
from ultralytics import YOLO
import yaml
import shutil

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    print("Environnement Google Colab détecté.")
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/Projets/AgbleDo_01') 
    os.chdir(PROJECT_ROOT)
    
except ImportError:
    IN_COLAB = False
    print("Environnement local détecté.")
    PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()
    os.chdir(PROJECT_ROOT)

print(f"Racine du projet : {PROJECT_ROOT}")

In [ ]:
if IN_COLAB:
    !pip install -q ultralytics mlflow

# Configuration de MLflow
print("Initialisation du tracking MLflow...")
mlflow.set_tracking_uri("file://" + str(PROJECT_ROOT / "mlruns"))
mlflow.set_experiment("agbledo-disease-detection")
print("✅ MLflow : prêt.")

## Configuration, Reprise et Lancement


In [ ]:
model_config_path = PROJECT_ROOT / "configs/model.yaml"
dataset_config_path = PROJECT_ROOT / "configs/dataset.yaml"

with open(str(model_config_path), "r") as f:
    hyp_config = yaml.safe_load(f)

# Gestion de la reprise
run_name = "agbledɔ01"
run_dir = PROJECT_ROOT / "runs/detect" / run_name
last_checkpoint = run_dir / "weights" / "last.pt"

if last_checkpoint.exists():
    print(f"🔄 Reprise de l'entraînement détectée depuis {last_checkpoint}")
    model = YOLO(last_checkpoint)
    resume_flag = True
else:
    print("▶️ Nouvel entraînement depuis le début.")
    model = YOLO(hyp_config["model"])
    resume_flag = False

In [ ]:
with mlflow.start_run(run_name=run_name):
    if not resume_flag:
        mlflow.log_params(hyp_config)

    # Entraînement
    results = model.train(
        data=str(dataset_config_path),
        project="runs/detect",
        name=run_name,
        resume=resume_flag,
        epochs=hyp_config["epochs"],
        patience=hyp_config["patience"],
        batch=hyp_config["batch"],
        imgsz=hyp_config["imgsz"],
        device=hyp_config["device"],
        optimizer=hyp_config["optimizer"],
        lr0=hyp_config["lr0"],
        lrf=hyp_config["lrf"],
        weight_decay=hyp_config["weight_decay"],
        hsv_h=hyp_config["hsv_h"],
        hsv_s=hyp_config["hsv_s"],
        hsv_v=hyp_config["hsv_v"],
        degrees=hyp_config["degrees"],
        translate=hyp_config["translate"],
        scale=hyp_config["scale"],
        fliplr=hyp_config["fliplr"],
        mosaic=hyp_config["mosaic"],
        deterministic=True,
        seed=42,
        save=True
    )

## Sauvegarde finale

In [ ]:
best_weight_path = run_dir / "weights" / "best.pt"
final_model_dir = PROJECT_ROOT / "models"
final_model_dir.mkdir(exist_ok=True)

if best_weight_path.exists():
    shutil.copy(best_weight_path, final_model_dir / "agbledɔ01.pt")
    print(f"Modèle sauvegardé avec succès dans : {final_model_dir / 'agbledɔ01.pt'}")
else:
    print(f"{best_weight_path} non trouvé.")